# Скрипт для обробки супутникових знімків

Ноутбук містить повний набір інструментів для завантаження та обробки геопросторових даних з **Sentinel-2** та **Landsat-8**.

**Основні етапи:**

1.  **Обробка даних Sentinel-2**: розпаковка, об'єднання каналів, репроєкція та обрізка.
2.  **Обробка даних Landsat-8**: розпаковка, паншарпенінг та аналіз якості.
3.  **Запуск** основного процесу з налаштованими параметрами.

## Імпорт бібліотек та налаштування

Ця клітинка завантажує всі необхідні програмні пакети (бібліотеки) для роботи. Сюди входять інструменти для роботи з файловою системою (`os`), геопросторовими даними (`gdal`, `rasterio`), математичними обчисленнями (`numpy`), візуалізацією (`matplotlib`) та завантаженням даних (`sentinelhub`). Також тут активується режим `%matplotlib inline` для відображення графіків безпосередньо в ноутбуці.

In [ ]:
import os
import subprocess
import zipfile
import tarfile
import glob
from osgeo import gdal, osr
import rasterio
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from rasterio.plot import show as rio_show
from sentinelhub import SentinelHubRequest, DataCollection, MimeType, CRS, BBox, SHConfig, Geometry

print("Бібліотеки успішно імпортовано.")

## Глобальні налаштування GDAL

Тут визначаються глобальні параметри для створення ефективних GeoTIFF файлів. Ці налаштування, такі як стиснення `LZW` та плиткова структура `TILED=YES`, застосовуватимуться до всіх растрових файлів, що створюються в процесі роботи, для зменшення їх розміру та прискорення доступу до даних.

In [ ]:
GDAL_TIFF_CREATION_OPTIONS = ["COMPRESS=LZW", "TILED=YES", "BIGTIFF=IF_SAFER"]

# Форматуємо опції у список аргументів для командного рядка
GDAL_CO_ARGS_LIST = []
for opt_val in GDAL_TIFF_CREATION_OPTIONS:
    GDAL_CO_ARGS_LIST.extend(["-co", opt_val])

print("Константи GDAL налаштовано.")

## Визначення допоміжних функцій

Цей блок містить дві основні допоміжні функції:
* **Візуалізація (`display_raster`):** Дозволяє швидко відобразити будь-яке растрове зображення з автоматичним налаштуванням контрасту для кращої видимості.
* **Робота з архівами (`unpack_archives`):** Автоматизує процес розпакування вихідних даних із архівів `.zip` або `.tar.gz`.

In [ ]:
def display_raster(raster_path, title="", bands_to_plot=None, cmap='gray', contrast_stretch_percentile=(2, 98), figsize=(8, 8)):
    """Візуалізує растрове зображення з автоматичним розтягуванням контрасту."""
    if not os.path.exists(raster_path):
        print(f"Помилка: файл не знайдено за шляхом {raster_path}")
        return

    with rasterio.open(raster_path) as src:
        fig, ax = plt.subplots(1, 1, figsize=figsize)
        ax.set_title(title if title else os.path.basename(raster_path))
        ax.set_axis_off()

        plot_data_list, display_rgb = [], False
        # Визначаємо, які канали відображати
        if bands_to_plot:
            if len(bands_to_plot) >= 3:
                plot_data_list = [src.read(b, masked=True).astype(np.float32) for b in bands_to_plot[:3]]
                display_rgb = True
            elif len(bands_to_plot) >= 1:
                plot_data_list = [src.read(bands_to_plot[0], masked=True).astype(np.float32)]
        elif src.count >= 3:
            plot_data_list = [src.read(b, masked=True).astype(np.float32) for b in [1, 2, 3]]
            display_rgb = True
        elif src.count == 1:
            plot_data_list = [src.read(1, masked=True).astype(np.float32)]

        # Розтягування контрасту для кращої візуалізації
        stretched_bands_for_display = []
        for band_data_orig in plot_data_list:
            band_data = band_data_orig.copy()
            valid_pixels = band_data[~np.ma.getmaskarray(band_data) & np.isfinite(band_data)]
            if src.nodata is not None: valid_pixels = valid_pixels[valid_pixels != src.nodata]
            if valid_pixels.size > 0:
                vmin, vmax = np.percentile(valid_pixels, contrast_stretch_percentile)
                if vmax <= vmin: vmin, vmax = valid_pixels.min(), valid_pixels.max()
                if vmax <= vmin: vmax = vmin + 1e-5
                band_data = np.clip(band_data, vmin, vmax)
                if (vmax - vmin) > 1e-6:
                    band_data = (band_data - vmin) / (vmax - vmin)
                if np.ma.is_masked(band_data): band_data = np.ma.filled(band_data, 0)
            stretched_bands_for_display.append(band_data)

        # Відображення
        if display_rgb and len(stretched_bands_for_display) == 3:
            rio_show(np.stack(stretched_bands_for_display, axis=0), ax=ax, transform=src.transform)
        elif stretched_bands_for_display:
            rio_show(stretched_bands_for_display[0], ax=ax, cmap=cmap, transform=src.transform)
        plt.show()

def unpack_archives(source_folder, target_folder_base, archive_type_filter=('.zip',)):
    """Розпаковує архіви (.zip, .tar.gz) з вихідної папки у цільову."""
    os.makedirs(target_folder_base, exist_ok=True)
    unpacked_product_paths = []
    for item in os.listdir(source_folder):
        item_lower = item.lower()
        archive_path = os.path.join(source_folder, item)
        archive_name_no_ext, archive_handler_type = "", None
        if item_lower.endswith('.zip') and '.zip' in archive_type_filter:
            archive_name_no_ext, archive_handler_type = os.path.splitext(item)[0], 'zip'
        else:
            for tar_pattern in ['.tar.gz', '.tar.bz2', '.tgz', '.tar']:
                if item_lower.endswith(tar_pattern) and tar_pattern in archive_type_filter:
                    archive_name_no_ext = item.replace(tar_pattern, "")
                    archive_handler_type = 'tar'
                    break
        if archive_handler_type:
            target_dir = os.path.join(target_folder_base, archive_name_no_ext)
            if os.path.exists(target_dir) and os.listdir(target_dir):
                unpacked_product_paths.append(target_dir)
                continue
            print(f"Розпакування {item}...")
            os.makedirs(target_dir, exist_ok=True)
            try:
                if archive_handler_type == 'zip':
                    with zipfile.ZipFile(archive_path, 'r') as zip_ref:
                        zip_ref.extractall(target_dir)
                elif archive_handler_type == 'tar':
                    with tarfile.open(archive_path, 'r:*') as tar_ref:
                        tar_ref.extractall(path=target_dir)
                unpacked_product_paths.append(target_dir)
            except Exception as e:
                print(f"  ПОМИЛКА під час розпакування {item}: {e}")
    return unpacked_product_paths

print("Функції візуалізації та роботи з архівами визначено.")

## Створення функцій-обгорток для GDAL

У цьому розділі ми створюємо прості функції Python, які "огортають" складні команди GDAL. Тут визначені функції для репроєкції, об'єднання, обрізки та паншарпенінгу зображень.

In [ ]:
def run_gdal_command(command_list):
    """Запускає команду GDAL та обробляє можливі помилки."""
    try:
        subprocess.run(command_list, check=True, capture_output=True, text=True, encoding='utf-8', errors='replace')
    except subprocess.CalledProcessError as e:
        print(f"    ПОМИЛКА ВИКОНАННЯ GDAL: Команда '{' '.join(e.cmd)}' повернула код {e.returncode}")
        if e.stdout: print(f"    GDAL STDOUT (вихід):\n{e.stdout}")
        if e.stderr: print(f"    GDAL STDERR (помилка):\n{e.stderr}")
        raise

def reproject_image_cmd(input_tiff, output_tiff, target_srs="EPSG:4326", resample_alg="bilinear"):
    """Репроєктує зображення за допомогою gdalwarp."""
    cmd = ["gdalwarp", "-t_srs", target_srs, "-r", resample_alg, "-multi"] + GDAL_CO_ARGS_LIST + ["-overwrite", input_tiff, output_tiff]
    run_gdal_command(cmd)

def merge_rasters_cmd(input_files_list, output_tiff, use_separate=False, extra_options=None):
    """Об'єднує кілька растрів в один за допомогою gdal_merge.py."""
    if not input_files_list: return
    cmd = ["python", "gdal_merge.py", "-o", output_tiff]
    if use_separate: cmd.append("-separate")
    cmd.extend(["-n", "-9999", "-a_nodata", "-9999"])
    if extra_options: cmd.extend(extra_options)
    cmd.extend(input_files_list)
    cmd.extend(GDAL_CO_ARGS_LIST)
    run_gdal_command(cmd)

def clip_by_vector_cmd(input_tiff, output_tiff, vector_shp):
    """Обрізає растр за векторним контуром за допомогою gdalwarp."""
    cmd = ["gdalwarp", "-cutline", vector_shp, "-crop_to_cutline", "-multi", "-dstnodata", "-9999"] + GDAL_CO_ARGS_LIST + ["-overwrite", input_tiff, output_tiff]
    run_gdal_command(cmd)

def pansharpen_image_gdal(pan_image, spectral_image, output_path, method, nodata_val=None, extra_opts=None):
    """Виконує паншарпенінг за допомогою gdal_pansharpen.py."""
    cmd = ["python", "gdal_pansharpen.py", "-r", method] + GDAL_CO_ARGS_LIST
    if nodata_val is not None: cmd.extend(["-nodata", str(nodata_val)])
    if extra_opts: cmd.extend(extra_opts)
    cmd.extend([pan_image, spectral_image, output_path])
    run_gdal_command(cmd)

print("Функції-обгортки для GDAL визначено.")

## Обробка даних Sentinel-2

Цей блок містить весь функціонал, необхідний для повного циклу обробки знімків Sentinel-2. Процес включає наступні кроки:
1.  **Завантаження даних** через API або пошук локальних архівів.
2.  **Об'єднання каналів** в єдиний багатоканальний файл.
3.  **Репроєкція** зображення у стандартну систему координат (WGS 84).
4.  **Об'єднання** кількох знімків в одну мозаїку.
5.  **Обрізка** фінального зображення за векторним контуром.

In [ ]:
def concatenate_bands_sentinel(product_folder, output_tiff_path, band_ids=["B02", "B03", "B04", "B08"]):
    """Знаходить та об'єднує вказані канали Sentinel-2 в один GeoTIFF."""
    safe_dir = next(iter(glob.glob(os.path.join(product_folder, "*.SAFE"))), product_folder)
    search_path = next(iter(glob.glob(os.path.join(safe_dir, "**", "IMG_DATA"), recursive=True)), None)
    if not search_path: return False

    all_jp2_files = glob.glob(os.path.join(search_path, "**", "*.jp2"), recursive=True)
    band_files_dict = {b_id: next((f for f in all_jp2_files if f"_{b_id}." in os.path.basename(f) or f"_{b_id}_" in os.path.basename(f)), None) for b_id in band_ids}

    ordered_band_paths = [band_files_dict[b] for b in band_ids if band_files_dict.get(b)]
    if len(ordered_band_paths) < len(band_ids): return False

    temp_vrt = os.path.join(os.path.dirname(output_tiff_path), f"temp_{os.path.basename(product_folder)}.vrt")
    gdal.BuildVRT(temp_vrt, ordered_band_paths, options=gdal.BuildVRTOptions(separate=True))
    gdal.Translate(output_tiff_path, temp_vrt, format="GTiff", creationOptions=GDAL_TIFF_CREATION_OPTIONS)
    if os.path.exists(temp_vrt): os.remove(temp_vrt)
    return True

def download_data_from_sentinel_hub_api(output_folder="sentinel_hub_downloads", config=None):
    """Завантажує RGB-зображення Sentinel-2 за вказаними параметрами через Sentinel Hub API."""
    print("\n===== ЗАВАНТАЖЕННЯ ДАНИХ ЧЕРЕЗ SENTINEL HUB API =====")

    if not config or not config.sh_client_id or not config.sh_client_secret:
        print("ПОПЕРЕДЖЕННЯ: Не надано конфігурацію або облікові дані Sentinel Hub. Пропуск завантаження.")
        return None

    evalscript = """//VERSION=3
    function setup() { return { input: ["B02", "B03", "B04"], output: { bands: 3 } }; }
    function evaluatePixel(sample) { return [2.5 * sample.B04, 2.5 * sample.B03, 2.5 * sample.B02]; }"""

    bbox_coords = [29.073321, 49.845775, 31.986007, 51.278667]
    bbox_obj = BBox(bbox=bbox_coords, crs=CRS.WGS84)
    img_width, img_height = 2500, 1907

    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[SentinelHubRequest.input_data(data_collection=DataCollection.SENTINEL2_L2A, time_interval=('2019-08-01', '2019-08-31'))],
        responses=[SentinelHubRequest.output_response('default', MimeType.TIFF)],
        bbox=bbox_obj, size=[img_width, img_height], config=config
    )

    try:
        print("  Надсилання запиту до Sentinel Hub API...")
        image_data = request.get_data()[0]
        date_str = request.get_dates()[0].strftime('%Y%m%d')
        output_filename = os.path.join(output_folder, f"s2_api_{date_str}_{img_width}x{img_height}.tif")
        transform = rasterio.transform.from_bounds(*bbox_obj, width=img_width, height=img_height)

        with rasterio.open(output_filename, 'w', driver='GTiff', height=img_height, width=img_width, count=3,
                           dtype=image_data.dtype, crs=f"EPSG:{bbox_obj.crs.value}", transform=transform,
                           compress='lzw', tiled=True) as dst:
            dst.write(np.moveaxis(image_data, -1, 0))
        print(f"  Завантаження успішне: {output_filename}")
        return output_filename
    except Exception as e:
        print(f"  ПОМИЛКА під час завантаження даних з Sentinel Hub: {e}")
        return None

def process_sentinel_data(source_folder, base_unpacked_folder, processed_folder, vector_contour_shp, enable_visualization, attempt_api_download=True, sh_config=None):
    """Основна функція для обробки даних Sentinel-2."""
    print("\n===== ОБРОБКА SENTINEL-2 =====")
    os.makedirs(processed_folder, exist_ok=True)
    reprojected_tiffs_for_merging = []
    data_source_type = "Offline"

    if attempt_api_download:
        api_output_folder = os.path.join(processed_folder, "api_downloads")
        os.makedirs(api_output_folder, exist_ok=True)
        downloaded_tiff = download_data_from_sentinel_hub_api(output_folder=api_output_folder, config=sh_config)
        if downloaded_tiff:
            data_source_type = "API"
            product_name = os.path.splitext(os.path.basename(downloaded_tiff))[0]
            reprojected_tiff = os.path.join(processed_folder, f"{product_name}_reprojected_4326.tif")
            reproject_image_cmd(downloaded_tiff, reprojected_tiff)
            if os.path.exists(reprojected_tiff): reprojected_tiffs_for_merging.append(reprojected_tiff)

    if not reprojected_tiffs_for_merging:
        print("=== Етап 1 (Офлайн): Розпаковка та обробка локальних архівів ===")
        unpacked_folders = unpack_archives(source_folder, base_unpacked_folder, archive_type_filter=('.zip',))
        if not unpacked_folders: print("Не знайдено архівів Sentinel-2 для оффлайн обробки."); return
        for product_path in unpacked_folders:
            product_name = os.path.basename(product_path)
            print(f"  Обробка продукту: {product_name}")
            concatenated = os.path.join(processed_folder, f"{product_name}_concatenated.tif")
            reprojected = os.path.join(processed_folder, f"{product_name}_reprojected_4326.tif")
            if concatenate_bands_sentinel(product_path, concatenated):
                reproject_image_cmd(concatenated, reprojected)
                if os.path.exists(reprojected): reprojected_tiffs_for_merging.append(reprojected)
                if os.path.exists(concatenated): os.remove(concatenated)

    if not reprojected_tiffs_for_merging: return

    print("=== Етап 2: Об'єднання та обрізка зображень ===")
    merged_tiff = os.path.join(processed_folder, f"sentinel_{data_source_type.lower()}_merged_4326.tif")
    merge_rasters_cmd(reprojected_tiffs_for_merging, merged_tiff)

    clipped_tiff = os.path.join(processed_folder, f"sentinel_{data_source_type.lower()}_final_clipped_4326.tif")
    display_target = merged_tiff

    if os.path.exists(merged_tiff) and os.path.exists(vector_contour_shp):
        clip_by_vector_cmd(merged_tiff, clipped_tiff, vector_contour_shp)
        display_target = clipped_tiff if os.path.exists(clipped_tiff) else merged_tiff

    if enable_visualization and os.path.exists(display_target):
        title_prefix = f"Sentinel-2 ({data_source_type}) {'Clipped' if display_target == clipped_tiff else 'Merged'}"
        if data_source_type == "API":
            display_raster(display_target, title=f"{title_prefix} (True Color)", bands_to_plot=[1, 2, 3])
        else:
            display_raster(display_target, title=f"{title_prefix} (True Color: B4,B3,B2)", bands_to_plot=[3, 2, 1])
            display_raster(display_target, title=f"{title_prefix} (False Color: B8,B4,B3)", bands_to_plot=[4, 3, 2])

print("Функції для Sentinel-2 визначено.")

## Обробка даних Landsat-8 та паншарпенінг

Цей розділ присвячений обробці знімків Landsat-8, головною метою якої є паншарпенінг — підвищення роздільної здатності. Процес включає:
1.  **Підготовку даних**: пошук необхідних каналів та створення еталонного RGB-зображення.
2.  **Тестування методів**: послідовне застосування різних алгоритмів паншарпенінгу.
3.  **Оцінка якості**: розрахунок статистичних метрик ($R^2$, MSE, MAE) для кожного методу.
4.  **Формування звіту**: вивід порівняльної таблиці результатів для вибору найкращого алгоритму.

In [ ]:
def find_landsat_bands(product_folder, band_suffixes_map, product_id_prefix=""):
    """Знаходить файли каналів Landsat за суфіксами."""
    band_files = {}
    all_files = glob.glob(os.path.join(product_folder, "**", "*.TIF"), recursive=True)
    for band_name, suffix in band_suffixes_map.items():
        search_pattern = (product_id_prefix if product_id_prefix else "") + "*" + suffix
        found_file = next((f for f in all_files if glob.fnmatch.fnmatch(os.path.basename(f), search_pattern)), None)
        if found_file: band_files[band_name] = found_file
    return band_files

def stack_landsat_bands_vrt(band_paths_ordered, output_vrt_path, common_nodata=None):
    """Створює віртуальний растр (VRT) з окремих каналів."""
    gdal.BuildVRT(output_vrt_path, band_paths_ordered, options=gdal.BuildVRTOptions(separate=True, VRTNodata=str(common_nodata) if common_nodata is not None else None))

def convert_vrt_to_tiff(input_vrt, output_tiff):
    """Конвертує VRT у GeoTIFF."""
    gdal.Translate(output_tiff, input_vrt, format="GTiff", creationOptions=GDAL_TIFF_CREATION_OPTIONS)

def resample_image_gdal_translate(input_raster, output_raster, x_res, y_res, resample_alg='bilinear'):
    """Змінює роздільну здатність растра за допомогою gdal_translate."""
    cmd = ["gdal_translate", "-tr", str(x_res), str(y_res), "-r", resample_alg] + GDAL_CO_ARGS_LIST + [input_raster, output_raster]
    run_gdal_command(cmd)

def calculate_pansharpening_metrics(original_rgb_path, pansharpened_rgb_path, nodata_val=None):
    """Розраховує метрики R2, MSE, MAE для оцінки якості паншарпенінгу."""
    metrics = {}
    with rasterio.open(original_rgb_path) as src_orig, rasterio.open(pansharpened_rgb_path) as src_pan:
        band_metrics_summary = {'r2': [], 'mse': [], 'mae': []}
        for i in range(1, src_orig.count + 1):
            orig_band, ps_band = src_orig.read(i, masked=True), src_pan.read(i, masked=True)
            mask = ~orig_band.mask & ~ps_band.mask
            orig_flat, ps_flat = orig_band[mask].flatten(), ps_band[mask].flatten()
            if orig_flat.size > 0:
                r2, mse, mae = r2_score(orig_flat, ps_flat), mean_squared_error(orig_flat, ps_flat), mean_absolute_error(orig_flat, ps_flat)
                metrics.update({f'band_{i}_r2': r2, f'band_{i}_mse': mse, f'band_{i}_mae': mae})
                for key, val in zip(['r2', 'mse', 'mae'], [r2, mse, mae]): band_metrics_summary[key].append(val)

        avg_r2 = np.nanmean(band_metrics_summary['r2']) if band_metrics_summary['r2'] else np.nan
        if not np.isnan(avg_r2):
            metrics.update({'avg_r2': avg_r2, 'avg_mse': np.nanmean(band_metrics_summary['mse']), 'avg_mae': np.nanmean(band_metrics_summary['mae'])})
    return metrics

def process_landsat_data(source_folder, base_unpacked_folder, processed_folder, product_ids, band_map, nodata_val, ps_methods, enable_visualization):
    """Основна функція для обробки даних Landsat-8 та виконання паншарпенінгу."""
    print("\n===== ОБРОБКА LANDSAT-8 ТА ПАНШАРПЕНІНГ =====")
    os.makedirs(processed_folder, exist_ok=True)
    unpacked_folders = unpack_archives(source_folder, base_unpacked_folder, archive_type_filter=('.tar.gz', '.tar', '.zip'))
    unpacked_to_process = [p for p in unpacked_folders if os.path.basename(p) in product_ids]
    if not unpacked_to_process: print(f"Не знайдено розпакованих папок для ID: {product_ids}."); return

    all_scenes_metrics = {}
    for product_path in unpacked_to_process:
        product_id = os.path.basename(product_path)
        print(f"\nОбробка продукту Landsat: {product_id}")
        scene_proc_folder = os.path.join(processed_folder, product_id); os.makedirs(scene_proc_folder, exist_ok=True)

        band_files = find_landsat_bands(product_path, band_map, product_id_prefix=product_id)
        if not all(b in band_files for b in ["B4", "B3", "B2", "PAN"]):
            print(f"  ПОПЕРЕДЖЕННЯ: Відсутні необхідні канали для {product_id}. Продукт пропущено."); continue

        truth_rgb_30m_tif = os.path.join(scene_proc_folder, f"{product_id}_RGB_30m_truth.tif")
        truth_rgb_30m_vrt = truth_rgb_30m_tif.replace(".tif", ".vrt")
        stack_landsat_bands_vrt([band_files["B4"], band_files["B3"], band_files["B2"]], truth_rgb_30m_vrt, common_nodata=nodata_val)
        convert_vrt_to_tiff(truth_rgb_30m_vrt, truth_rgb_30m_tif)
        if enable_visualization: display_raster(truth_rgb_30m_tif, title=f"L8 Ground Truth: {product_id}")

        pan_15m_resampled = os.path.join(scene_proc_folder, f"{product_id}_PAN_resampled_30m.tif")
        resample_image_gdal_translate(band_files["PAN"], pan_15m_resampled, 30, -30, resample_alg='cubic')

        rgb_30m_resampled = os.path.join(scene_proc_folder, f"{product_id}_RGB_resampled_60m.tif")
        resample_image_gdal_translate(truth_rgb_30m_tif, rgb_30m_resampled, 60, -60, resample_alg='average')

        scene_metrics = {}
        print("  Тестування методів паншарпенінгу...")
        for ps_method in ps_methods:
            pansharpened_tif = os.path.join(scene_proc_folder, f"{product_id}_pansharpened_30m_{ps_method}.tif")
            pansharpen_image_gdal(pan_15m_resampled, rgb_30m_resampled, pansharpened_tif, ps_method, nodata_val=nodata_val, extra_opts=["-threads", "ALL_CPUS"])
            if os.path.exists(pansharpened_tif):
                metrics = calculate_pansharpening_metrics(truth_rgb_30m_tif, pansharpened_tif, nodata_val=nodata_val)
                scene_metrics[ps_method] = metrics if metrics else {}
        all_scenes_metrics[product_id] = {"metrics": scene_metrics, "folder": scene_proc_folder}

    # ДЕТАЛЬНИЙ ЗВІТ ПРО РЕЗУЛЬТАТИ
    print("\n" + "="*20 + " РЕЗУЛЬТАТИ ПАНШАРПЕНІНГУ " + "="*20)
    best_overall_method, highest_overall_r2 = None, -float('inf')
    for scene_id, data in all_scenes_metrics.items():
        print(f"\nРезультати для сцени: {scene_id}")
        best_method_scene, highest_r2_scene = None, -float('inf')
        for method, m_vals in sorted(data["metrics"].items(), key=lambda item: item[1].get('avg_r2', -1), reverse=True):
            avg_r2 = m_vals.get('avg_r2', np.nan)
            if not np.isnan(avg_r2):
                print(f"    --- Метод: {method} ---")
                print(f"      Середні:   R²: {avg_r2:7.4f} | MSE: {m_vals.get('avg_mse', np.nan):8.2f} | MAE: {m_vals.get('avg_mae', np.nan):8.2f}")
                if avg_r2 > highest_r2_scene: highest_r2_scene, best_method_scene = avg_r2, method
                if avg_r2 > highest_overall_r2: highest_overall_r2, best_overall_method = avg_r2, method
        if best_method_scene and enable_visualization:
            print(f"\n  Найкращий метод для '{scene_id}': {best_method_scene} (R² = {highest_r2_scene:.4f})")
            best_ps_file = os.path.join(data["folder"], f"{scene_id}_pansharpened_30m_{best_method_scene}.tif")
            if os.path.exists(best_ps_file): display_raster(best_ps_file, title=f"Best PS ({best_method_scene}): {scene_id}")
    if best_overall_method: print(f"\nЗагалом найкращий метод: {best_overall_method} (R² = {highest_overall_r2:.4f})")

print("Функції для Landsat-8 та паншарпенінгу визначено.")

## Конфігурація та запуск основного процесу

Це головна керуюча клітинка. Тут зібрані всі налаштування, які можна змінювати:
* **Перемикачі** для активації/деактивації окремих етапів обробки.
* **Облікові дані** для доступу до Sentinel Hub API.
* **Параметри обробки**, такі як шляхи до папок, ID знімків та список методів паншарпенінгу.

Після налаштування параметрів, запуск цієї клітинки ініціює весь процес обробки даних.

In [ ]:
# --- ГОЛОВНІ ПЕРЕМИКАЧІ ---
ENABLE_VISUALIZATION = True       # Показувати зображення в процесі обробки
RUN_SENTINEL_PROCESSING = True    # Запускати обробку Sentinel-2
ATTEMPT_S2_API_DOWNLOAD = False   # True: спробувати завантажити дані через API. False: використовувати лише локальні файли.
RUN_LANDSAT_PROCESSING = True     # Запускати обробку Landsat-8

# --- ПАРАМЕТРИ ДЛЯ SENTINEL HUB API ---
# ВАЖЛИВО: Вставте свої облікові дані Sentinel Hub тут, або встановіть їх як змінні середовища
sh_config = SHConfig()
sh_config.sh_client_id = os.environ.get('SH_CLIENT_ID', 'c728f23d-a45e-469a-813f-dc04d9a2ceea')
sh_config.sh_client_secret = os.environ.get('SH_CLIENT_SECRET', 'jrMt44QBBZllusjodpjf975tXH6GXB7M')

# --- ПАРАМЕТРИ ДЛЯ SENTINEL-2 ---
s2_params = {
    "source_folder": "sentinel_raw_data",
    "base_unpacked_folder": "unpacked_sentinel_data",
    "processed_folder": "processed_sentinel_data",
    "vector_contour_shp": "Kyiv_regions.shp",
    "enable_visualization": ENABLE_VISUALIZATION,
    "attempt_api_download": ATTEMPT_S2_API_DOWNLOAD,
    "sh_config": sh_config
}

# --- ПАРАМЕТРИ ДЛЯ LANDSAT-8 ---
l8_params = {
    "source_folder": "landsat_raw_data",
    "base_unpacked_folder": "unpacked_landsat_data",
    "processed_folder": "processed_landsat_data",
    "product_ids": [
        "LC08_L1TP_182025_20190830_20190903_01_T1",
        "LC08_L1TP_182026_20190830_20190903_01_T1"
    ],
    "band_map": {"B2": "_B2.TIF", "B3": "_B3.TIF", "B4": "_B4.TIF", "PAN": "_B8.TIF"},
    "nodata_val": 0,
    "ps_methods": ['nearest', 'bilinear', 'cubic', 'cubicspline', 'lanczos', 'average'],
    "enable_visualization": ENABLE_VISUALIZATION
}

# --- Створення папок та запуск ---
print("Параметри налаштовано. Створюю папки та починаю обробку...")

for params_dict in [s2_params, l8_params]:
    for folder_key in ["source_folder", "base_unpacked_folder", "processed_folder"]:
        if folder_key in params_dict and params_dict[folder_key]:
            os.makedirs(params_dict[folder_key], exist_ok=True)

if RUN_SENTINEL_PROCESSING:
    process_sentinel_data(**s2_params)

if RUN_LANDSAT_PROCESSING:
    process_landsat_data(**l8_params)

print(f"\nОбробку завершено. Перевірте папки '{s2_params['processed_folder']}' та '{l8_params['processed_folder']}' для результатів.")